# Listings data preparation 
In this notebook we prepare Airbnb listings dataset for predictive modeling. We aim to identify the variables that may explain listing prices in Rome and clean the dataset. 
Our target variable selected for this stage is the lsiting nightly price.

In [70]:
#setup
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: /Users/antoniacordova/Data for business/airbnb-rome-analysis


## 1. Load dataset

In [71]:
listings = pd.read_csv(
    PROJECT_ROOT / "data" / "listings.csv.gz",
    compression="gzip"
)
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2737,https://www.airbnb.com/rooms/2737,20250914152919,2025-09-15,city scrape,"Elif's room in cozy, clean flat.",10 min by bus you can get to Piazza Venezia or...,It used to be an industrial area until late 80...,https://a0.muscache.com/pictures/41225252/e955...,3047,...,5.00,4.40,4.40,NaN,f,6,0,6,0,0.04
1,11834,https://www.airbnb.com/rooms/11834,20250914152919,2025-09-15,city scrape,"Charming Boschetto Studio, Rome",Fantastic apartment in the Monti district. The...,"""Monti"" with its narrow cobblestone alleys, cr...",https://a0.muscache.com/pictures/miso/Hosting-...,44552,...,4.96,4.99,4.81,IT058091C29VJSIZQZ,f,1,1,0,0,1.62
2,12398,https://www.airbnb.com/rooms/12398,20250914152919,2025-09-15,city scrape,Casa Donatello - Home far from Home,Casa Donatello is a newly renovated two-bedroo...,You are at 15 minutes walking distance from hi...,https://a0.muscache.com/pictures/miso/Hosting-...,11756,...,5.00,4.89,4.83,it058091c2kv6epw8f,f,1,1,0,0,0.47
3,19965,https://www.airbnb.com/rooms/19965,20250914152919,2025-09-15,city scrape,S. Peter's Square 5 Min WALK bright and quite ...,AT ONLY 5 MINUTES WALK to S.Peter's Basilica S...,Prati is a famous neighbourhood (rione of Rome...,https://a0.muscache.com/pictures/hosting/Hosti...,75450,...,4.90,4.90,4.54,IT058091C20YD35BX2,t,3,3,0,0,1.07
4,19967,https://www.airbnb.com/rooms/19967,20250914152919,2025-09-15,city scrape,*In front Vatican Museums 2 bedrooms quite bri...,"IN FRONT of the Vatican Museums entrance, at O...",Prati is a famous neighbourhood (rione of Rome...,https://a0.muscache.com/pictures/hosting/Hosti...,75450,...,4.80,4.85,4.28,IT058091C20YD35BX2,t,3,3,0,0,0.32


## 2. Exploration

In [72]:
listings.shape

(37652, 79)

In [73]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 37652 entries, 0 to 37651
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            37652 non-null  int64  
 1   listing_url                                   37652 non-null  str    
 2   scrape_id                                     37652 non-null  int64  
 3   last_scraped                                  37652 non-null  str    
 4   source                                        37652 non-null  str    
 5   name                                          37652 non-null  str    
 6   description                                   36703 non-null  str    
 7   neighborhood_overview                         17721 non-null  str    
 8   picture_url                                   37652 non-null  str    
 9   host_id                                       37652 non-null  int64  
 1

In [74]:
listings.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

## 4. Dataset cleaning
In the dataset the `price` variable is stored as text and contains currency symbols.
To use it in our model, it must be converted into a numeric format. 

In [75]:
listings["price"].head()

0     $57.00
1    $110.00
2    $124.00
3    $162.00
4    $150.00
Name: price, dtype: str

In [76]:
listings["price"].isna().sum()

np.int64(4088)

In [77]:
listings["price"] = (
    listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [78]:
# to verify
listings["price"].head()

0     57.0
1    110.0
2    124.0
3    162.0
4    150.0
Name: price, dtype: float64

Feature selection was guided by the adquire knowledge from Airbnb official webpage about marketplace, prioritazing location variables, accomodation characteristic, host reputation, and guest satisfaction. 
 ## 5. Selection of features
 Having in consideration the complete list of features, we decided to delete the following ones: 
 #### 5.1. Identifiers
 these variables identify the records but they do not contain  information that explains the prices of each residency. Including them would just add noise to the model. 
 #### 5.2. Textual description 
 This group of variables might contain useful information but we need NLP for their analysis. Therefore, we will also exclude them. 
 #### 5.3. Image and media
 Again, is not numerical useful information, it goes beyond our lecture. 
 #### 5.4. Administrative fields
 Variables that describe how the information was retrieved does not affect the listing price.
 #### 5.5. Redundant variables
In this group there are variable that have very similar information between them, if we were to include them we would only create redundance in our models. 
#### 5.6. Reviews
For the first part of our work we will exclude them, but they might be more useful when we apply feature engineering. 

In [79]:
candidate_features = [
    "price",

    "neighbourhood_cleansed",

    "latitude",
    "longitude",

    "property_type",
    "room_type",

    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",

    "host_is_superhost",

    "host_response_rate",
    "host_acceptance_rate",

    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",

    "number_of_reviews",
    "reviews_per_month",

    "minimum_nights",
    "maximum_nights",

    "instant_bookable"
]

In [80]:
selected = listings[candidate_features]

selected.head()

,price,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,bathrooms,...,host_acceptance_rate,review_scores_rating,review_scores_cleanliness,review_scores_location,review_scores_value,number_of_reviews,reviews_per_month,minimum_nights,maximum_nights,instant_bookable
0,57.0,VIII Appia Antica,41.871360,12.482150,Private room,Private room,1,1.0,1.0,1.5,...,0%,4.80,4.60,4.40,4.40,5,0.04,31,1125,f
1,110.0,I Centro Storico,41.895447,12.491181,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,...,95%,4.86,4.93,4.99,4.81,284,1.62,2,28,f
2,124.0,II Parioli/Nomentano,41.925820,12.469280,Entire rental unit,Entire home/apt,6,2.0,3.0,1.0,...,100%,4.90,4.90,4.89,4.83,85,0.47,3,365,f
3,162.0,I Centro Storico,41.908230,12.452930,Entire condo,Entire home/apt,5,2.0,3.0,1.0,...,99%,4.57,4.46,4.90,4.54,178,1.07,3,365,t
4,150.0,I Centro Storico,41.908283,12.452617,Entire rental unit,Entire home/apt,5,2.0,3.0,1.0,...,99%,4.28,4.02,4.85,4.28,46,0.32,3,365,t


In [81]:
selected = selected.dropna(subset=["price"])

## 6. Missing values analysis
This is an important step to comprehend the information that our model carries and therefore after being able to realise a proper selection of variables. 

In [82]:
missing_table = pd.DataFrame({
    "missing_count": selected.isna().sum(),
    "missing_pct": selected.isna().mean() * 100
})

missing_table = (
    missing_table
    .sort_values("missing_pct", ascending=False)
)

missing_table

,missing_count,missing_pct
host_response_rate,4737,14.113336
review_scores_location,4297,12.802407
review_scores_value,4297,12.802407
review_scores_cleanliness,4296,12.799428
review_scores_rating,4295,12.796449
reviews_per_month,4295,12.796449
host_acceptance_rate,2432,7.245859
host_is_superhost,2034,6.060064
bedrooms,31,0.092361
beds,17,0.050650


In [83]:
missing_table[missing_table["missing_count"] > 0]

,missing_count,missing_pct
host_response_rate,4737,14.113336
review_scores_location,4297,12.802407
review_scores_value,4297,12.802407
review_scores_cleanliness,4296,12.799428
review_scores_rating,4295,12.796449
reviews_per_month,4295,12.796449
host_acceptance_rate,2432,7.245859
host_is_superhost,2034,6.060064
bedrooms,31,0.092361
beds,17,0.050650


## 7. Correction of percentage variables
`host_response_rate`
`host_acceptance_rate`
are stored as percentages and must be converted into numerical values.

In [84]:
selected["host_response_rate"].head()

0      0%
1    100%
2     NaN
3    100%
4    100%
Name: host_response_rate, dtype: str

In [85]:
selected["host_acceptance_rate"].head()

0      0%
1     95%
2    100%
3     99%
4     99%
Name: host_acceptance_rate, dtype: str

In [86]:
selected["host_response_rate"] = (
    selected["host_response_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

In [87]:
selected["host_acceptance_rate"] = (
    selected["host_acceptance_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

In [88]:
selected["host_response_rate"] = pd.to_numeric(
    selected["host_response_rate"],
    errors="coerce"
)

selected["host_acceptance_rate"] = pd.to_numeric(
    selected["host_acceptance_rate"],
    errors="coerce"
)

In [89]:
selected[["host_response_rate", "host_acceptance_rate"]].describe()

,host_response_rate,host_acceptance_rate
count,28827.000000,31132.000000
mean,96.758005,93.019466
std,13.471273,19.524055
min,0.000000,0.000000
25%,100.000000,98.000000
50%,100.000000,100.000000
75%,100.000000,100.000000
max,100.000000,100.000000


## 8. Imputation strategy 
For the following numerical variables we will impute the median:

- bedrooms
- beds
- bathrooms
- host_response_rate
- host_acceptance_rate
- review_scores_rating
- review_scores_cleanliness
- review_scores_location
- review_scores_value
- reviews_per_month

Median imputation is robust to extreme values and preserves the  distribution better than the mean.

In [90]:
numerical_columns = [
    "bedrooms",
    "beds",
    "bathrooms",
    "host_response_rate",
    "host_acceptance_rate",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month"
]

for col in numerical_columns:
    selected[col] = selected[col].fillna(
        selected[col].median()
    )

For the categorical variables, we will simply impute the value "Unkwown"

- property_type
- room_type
- host_is_superhost
- instant_bookable

This allows us to still preserve the information about missingness

In [91]:
categorical_columns = [
    "property_type",
    "room_type",
    "host_is_superhost",
    "instant_bookable"
]

for col in categorical_columns:
    selected[col] = selected[col].fillna("Unknown")

In [92]:
# verification
selected.isna().sum()

price                        0
neighbourhood_cleansed       0
latitude                     0
longitude                    0
property_type                0
room_type                    0
accommodates                 0
bedrooms                     0
beds                         0
bathrooms                    0
host_is_superhost            0
host_response_rate           0
host_acceptance_rate         0
review_scores_rating         0
review_scores_cleanliness    0
review_scores_location       0
review_scores_value          0
number_of_reviews            0
reviews_per_month            0
minimum_nights               0
maximum_nights               0
instant_bookable             0
dtype: int64